# 02 — Data Cleaning & Integrity Verification
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Step:** Inspect and clean raw OHLCV data. Generate a data-quality report.

**Checks performed:**
1. Missing values (NaN / null)
2. Duplicate date entries
3. Invalid OHLC relationships
4. Zero or negative prices
5. Negative / zero volume anomalies
6. Chronological ordering

No records are silently discarded — every action is logged.

## 1. Imports & Paths

In [ ]:
import os
import sys
import logging
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

RAW_DATA_DIR  = os.path.join("data", "raw")
REPORT_DIR    = os.path.join("reports", "data_preparation")
REPORT_PATH   = os.path.join(REPORT_DIR, "data_quality_report.md")

STOCK_FILES = {
    "RELIANCE": "RELIANCE.csv",
    "TCS":      "TCS.csv",
    "INFY":     "INFY.csv",
    "HDFCBANK": "HDFCBANK.csv",
    "ITC":      "ITC.csv",
}

## 2. StockDataAuditor Class

In [ ]:
class StockDataAuditor:
    """Performs audit checks and data cleaning on stock market time series data."""

    def __init__(self, symbol: str, df: pd.DataFrame):
        self.symbol = symbol
        self.raw_df = df.copy()
        self.cleaned_df = df.copy()
        self.audit_log: Dict[str, Any] = {
            "symbol": symbol,
            "raw_row_count": len(df),
            "cleaned_row_count": 0,
            "raw_start_date": None,
            "raw_end_date": None,
            "missing_values_before": {},
            "missing_values_after": {},
            "duplicate_dates_found": 0,
            "invalid_ohlc_rows": 0,
            "non_positive_prices": 0,
            "negative_volumes": 0,
            "zero_volumes": 0,
            "chronological_disorders": 0,
            "actions_taken": [],
        }

    def audit_and_clean(self) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """Executes the full cleaning and verification pipeline."""
        df = self.cleaned_df

        # 1. Date normalisation and chronological check
        if "Date" not in df.columns:
            raise ValueError(f"[{self.symbol}] Missing 'Date' column.")
        df["Date"] = pd.to_datetime(df["Date"])
        self.audit_log["raw_start_date"] = df["Date"].min().strftime("%Y-%m-%d")
        self.audit_log["raw_end_date"]   = df["Date"].max().strftime("%Y-%m-%d")

        # Duplicate dates
        dup_mask  = df["Date"].duplicated()
        dup_count = int(dup_mask.sum())
        self.audit_log["duplicate_dates_found"] = dup_count
        if dup_count > 0:
            dup_dates = df.loc[dup_mask, "Date"].astype(str).tolist()
            logger.warning(f"[{self.symbol}] {dup_count} duplicate dates: {dup_dates}")
            df = df.drop_duplicates(subset=["Date"], keep="first")
            self.audit_log["actions_taken"].append(
                f"Removed {dup_count} duplicate date rows, keeping first observation."
            )

        # Chronological ordering
        if not df["Date"].is_monotonic_increasing:
            self.audit_log["chronological_disorders"] += 1
            logger.warning(f"[{self.symbol}] Data not in chronological order. Sorting...")
            df = df.sort_values("Date").reset_index(drop=True)
            self.audit_log["actions_taken"].append("Sorted chronologically by Date.")
        else:
            df = df.reset_index(drop=True)

        # 2. Coerce numerics and check missing values
        numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
        for col in numeric_cols:
            if col not in df.columns:
                raise ValueError(f"[{self.symbol}] Missing column '{col}'")
            df[col] = pd.to_numeric(df[col], errors="coerce")

        missing_before = df[numeric_cols].isna().sum().to_dict()
        self.audit_log["missing_values_before"] = {k: int(v) for k, v in missing_before.items()}

        total_missing = sum(missing_before.values())
        if total_missing > 0:
            logger.warning(f"[{self.symbol}] Missing values: {missing_before}")
            df[numeric_cols] = df[numeric_cols].ffill().bfill()
            self.audit_log["actions_taken"].append(
                f"Imputed {total_missing} missing values using ffill → bfill."
            )

        # 3. Invalid OHLC relationships
        inv = (
            (df["High"] < df["Low"]) |
            (df["Open"] > df["High"]) | (df["Open"] < df["Low"]) |
            (df["Close"] > df["High"]) | (df["Close"] < df["Low"])
        )
        inv_count = int(inv.sum())
        self.audit_log["invalid_ohlc_rows"] = inv_count
        if inv_count > 0:
            logger.error(f"[{self.symbol}] {inv_count} invalid OHLC rows at indices {df.index[inv].tolist()}")
            df.loc[inv, "High"] = df.loc[inv, ["Open", "High", "Low", "Close"]].max(axis=1)
            df.loc[inv, "Low"]  = df.loc[inv, ["Open", "High", "Low", "Close"]].min(axis=1)
            self.audit_log["actions_taken"].append(
                f"Repaired {inv_count} invalid OHLC rows: High=max(OHLC), Low=min(OHLC)."
            )

        # 4. Zero or negative prices
        non_pos = (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)
        non_pos_count = int(non_pos.sum())
        self.audit_log["non_positive_prices"] = non_pos_count
        if non_pos_count > 0:
            logger.error(f"[{self.symbol}] {non_pos_count} non-positive price rows.")
            df.loc[non_pos, ["Open", "High", "Low", "Close"]] = np.nan
            df[["Open", "High", "Low", "Close"]] = df[["Open", "High", "Low", "Close"]].ffill().bfill()
            self.audit_log["actions_taken"].append(
                f"Forward-filled {non_pos_count} non-positive price rows."
            )

        # 5. Volume anomalies
        neg_vol = (df["Volume"] < 0)
        neg_count = int(neg_vol.sum())
        self.audit_log["negative_volumes"] = neg_count
        if neg_count > 0:
            logger.warning(f"[{self.symbol}] {neg_count} negative volume entries — using abs().")
            df["Volume"] = df["Volume"].abs()
            self.audit_log["actions_taken"].append(f"Corrected {neg_count} negative volumes to positive.")

        zero_vol = (df["Volume"] == 0)
        zero_count = int(zero_vol.sum())
        self.audit_log["zero_volumes"] = zero_count
        if zero_count > 0:
            min_pos_vol = df.loc[df["Volume"] > 0, "Volume"].min()
            fill_vol = min_pos_vol if pd.notna(min_pos_vol) and min_pos_vol > 0 else 1.0
            df.loc[zero_vol, "Volume"] = fill_vol
            self.audit_log["actions_taken"].append(
                f"Replaced {zero_count} zero volumes with minimum positive volume ({fill_vol})."
            )

        # Final state
        missing_after = df[numeric_cols].isna().sum().to_dict()
        self.audit_log["missing_values_after"] = {k: int(v) for k, v in missing_after.items()}
        df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
        self.audit_log["cleaned_row_count"] = len(df)

        if not self.audit_log["actions_taken"]:
            self.audit_log["actions_taken"].append("Data verified clean — no anomalies detected.")

        self.cleaned_df = df
        return df, self.audit_log

## 3. Run Audit on All Stocks

In [ ]:
print("=" * 60)
print("STEP 2: Data Cleaning & Integrity Verification")
print("=" * 60)

cleaned_data: Dict[str, pd.DataFrame] = {}
audit_results: List[Dict[str, Any]] = []

for symbol, filename in STOCK_FILES.items():
    raw_path = os.path.join(RAW_DATA_DIR, filename)
    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Raw file missing: {raw_path} — run 01_data_collection.ipynb first.")

    raw_df  = pd.read_csv(raw_path)
    auditor = StockDataAuditor(symbol, raw_df)
    clean_df, audit_log = auditor.audit_and_clean()

    cleaned_data[symbol] = clean_df
    audit_results.append(audit_log)

    logger.info(
        f"[{symbol}] rows: {audit_log['raw_row_count']} → {audit_log['cleaned_row_count']} | "
        f"missing: {sum(audit_log['missing_values_before'].values())} | "
        f"dup_dates: {audit_log['duplicate_dates_found']} | "
        f"bad_ohlc: {audit_log['invalid_ohlc_rows']}"
    )

print("\nData cleaning complete.")

## 4. Display Audit Summary

In [ ]:
import pandas as pd

rows = []
for log in audit_results:
    rows.append({
        "Symbol":        log["symbol"],
        "Raw Rows":      log["raw_row_count"],
        "Clean Rows":    log["cleaned_row_count"],
        "Start Date":    log["raw_start_date"],
        "End Date":      log["raw_end_date"],
        "Missing":       sum(log["missing_values_before"].values()),
        "Dup Dates":     log["duplicate_dates_found"],
        "Bad OHLC":      log["invalid_ohlc_rows"],
        "Non-pos Px":    log["non_positive_prices"],
        "Zero/Neg Vol":  f"{log['zero_volumes']} / {log['negative_volumes']}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

## 5. Generate & Save Data Quality Report

In [ ]:
def generate_markdown_report(audit_results: List[Dict[str, Any]]) -> str:
    lines = [
        "# Data Quality & Integrity Report",
        "",
        "**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  ",
        "**Dataset:** National Stock Exchange of India (NSE) — 5 Target Equities  ",
        "**Source:** Yahoo Finance (yfinance)  ",
        "**Lookback Window:** 5 Years Historical Daily Data  ",
        "",
        "---",
        "",
        "## 1. Executive Summary",
        "",
        "A rigorous data hygiene and structural integrity check was conducted across all downloaded stocks.",
        "The validation protocol evaluated:",
        "1. **Missing values (NaN / Null)** across Open, High, Low, Close, and Volume.",
        "2. **Duplicate timestamps** and calendar continuity.",
        "3. **Physical OHLC validity:** `High >= Low`, `High >= max(Open, Close)`, `Low <= min(Open, Close)`.",
        "4. **Price positivity:** `Open, High, Low, Close > 0`.",
        "5. **Volume anomalies:** negative volume detection and zero-volume adjustments.",
        "6. **Chronological monotonic ordering**.",
        "",
        "---",
        "",
        "## 2. Stock-by-Stock Quality Audit Table",
        "",
        "| Symbol | Raw Rows | Cleaned Rows | Start Date | End Date | Missing Values | Duplicate Dates | Invalid OHLC | Non-Positive Prices | Zero / Neg Volumes |",
        "|---|---|---|---|---|---|---|---|---|---|",
    ]

    for log in audit_results:
        missing = sum(log["missing_values_before"].values())
        vol     = f"{log['zero_volumes']} zero, {log['negative_volumes']} neg"
        lines.append(
            f"| **{log['symbol']}** | {log['raw_row_count']} | {log['cleaned_row_count']} | "
            f"{log['raw_start_date']} | {log['raw_end_date']} | {missing} | "
            f"{log['duplicate_dates_found']} | {log['invalid_ohlc_rows']} | "
            f"{log['non_positive_prices']} | {vol} |"
        )

    lines.extend([
        "",
        "---",
        "",
        "## 3. Data Cleaning Actions & Academic Methodology",
        "",
    ])
    for log in audit_results:
        lines.append(f"### {log['symbol']}")
        for action in log["actions_taken"]:
            lines.append(f"- {action}")
        lines.append("")

    lines.extend([
        "---",
        "",
        "## 4. Verification Statement",
        "",
        "- **Zero Silently Discarded Records:** All adjustments were explicitly logged and audited.",
        "- **Chronological Integrity:** Every series is strictly monotonic, sorted from past to present.",
        "- **Academic Justification:** Price carry-forward (ffill) preserves the real market state where "
        "untraded assets retain their last settled price until subsequent market auctions.",
        "",
    ])
    return "\n".join(lines)


os.makedirs(REPORT_DIR, exist_ok=True)
report_md = generate_markdown_report(audit_results)
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report_md)
print(f"Data quality report saved → {REPORT_PATH}")